In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

import keras
import re


I0000 00:00:1782479455.283449   44614 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [63]:
# Chemin vers le fichier Excel
file_path = "selection_rod.xlsx"

# Lecture de la première feuille Excel
# header=None pour garder les lignes telles quelles
df_rod = pd.read_excel(file_path, sheet_name=0, header=None)

# Ligne 1 = indicateurs (1 = garder)
keep_mask = df_rod.iloc[0].fillna(0).astype(str).str.strip() == "1"

# Ligne 2 = noms de colonnes
columns = df_rod.iloc[1]

# Données à partir de la ligne 3
df_rod_clean = df_rod.iloc[2:, keep_mask.values].copy()

# Appliquer les noms de colonnes uniquement sur les colonnes gardées
df_rod_clean.columns = columns[keep_mask].values

# Réinitialiser l'index
df_rod_clean.reset_index(drop=True, inplace=True)

df_rod_clean = df_rod_clean.replace({
    "OUI": 1,
    "X": 1,
    "NON": 0,
    "-": 0,
    " - " : 0,
    "": np.nan,
    #np.nan: 0
})

# Remplacer les "?" par la moyenne de la colonne
for col in df_rod_clean.columns:
    mask_question = df_rod_clean[col].astype(str).str.strip() == "?"

    if mask_question.any():
        serie_numeric = pd.to_numeric(
            df_rod_clean[col].replace("?", np.nan),
            errors="coerce"
        )

        moyenne = serie_numeric.mean()

        df_rod_clean.loc[mask_question, col] = moyenne

# Remplacer les NaN restants par 0
df_rod_clean = df_rod_clean.fillna(0)

# df_marques = pd.get_dummies(df_rod_clean["MARQUE"], dtype = int, prefix="MARQUE", prefix_sep = " ")
# df_rod_clean[df_marques.columns] = df_marques

dummy_columns = ["MARQUE", "PMS"]
df_dummies = pd.get_dummies(df_rod_clean, columns = dummy_columns, dtype = int, prefix_sep = " ")
df_rod_clean[df_dummies.columns] = df_dummies
df_rod_clean = df_rod_clean.drop(dummy_columns, axis = 1)

df_rod_clean.rename(columns={"LATITUDE": "HOTEL_LAT", "LONGITUDE" : "HOTEL_LON", "NOM DE L'HOTEL" : "HOTEL_NAME", "CODE H" : "HOTEL_CODE"}, inplace=True)

/tmp/ipykernel_21865/2665346326.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  keep_mask = df_rod.iloc[0].fillna(0).astype(str).str.strip() == "1"
/tmp/ipykernel_21865/2665346326.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_rod_clean = df_rod_clean.replace({
/tmp/ipykernel_21865/2665346326.py:39: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_opti

In [64]:
df_rod_clean

,HOTEL_CODE,HOTEL_NAME,HOTEL_LON,HOTEL_LAT,NB. DE CHAMBRES,CONTRAT SIGNE (ANNEE),DERNIERE RENO HOTEL,DERNIERE RENO LOBBY,TO ANNUEL,TO LE PLUS BAS (TAUX),TO LE PLUS HAUT (TAUX),BAR,RESTAURANT,SALLES DE REUNION,SALLE DE SPORT,PISCINE,VITRINE RÉFRIGÉRÉE,MICRO-ONDES,FONTAINE À EAU,MACHINE À CAFÉ,BOUILLOIRE,LOISIRS (%),TOP 1 > COUPLES,TOP 1 > AMIS,TOP 1 > FAMILLES,AFFAIRES (%),NATIONAL (%),INTERNATIONAL (%),MÈTRES LINÉAIRES DEDIES A VOTRE CORNER (ACTUEL OU FUTUR),MARQUE IBIS BUDGET,MARQUE IBIS STYLES,MARQUE MERCURE,MARQUE NOVOTEL,PMS FOLS,PMS OPERA
0,H2075,IBIS BUDGET NICE CALIFORNIE,7.240512,43.689186,129,2003.666667,2020.0,2020.0,0.825,0.755,0.89,0,0,4,0,0,1,1,1,1,0.166667,70.0,0,1,0,30.0,65.0,35.0,6,1,0,0,0,1,0
1,HB6A3,IBIS BUDGET STRASBOURG REPUBLIQUE,7.754599,48.591522,97,2020.000000,2025.0,2025.0,0.700,0.600,0.80,0,0,0,0,0,0,0,1,1,0.000000,60.0,0,1,0,40.0,60.0,40.0,2,1,0,0,0,0,1
2,H0815,IBIS STYLES ROISSY CDG,2.519843,49.006733,309,2015.000000,2015.0,2015.0,0.950,0.910,0.98,1,1,1,0,0,0,1,1,1,1.000000,80.0,0,0,1,20.0,70.0,30.0,5,0,1,0,0,1,0
3,HB5I0,NOVOTEL MEGEVE MONT BLANC,6.619055,45.859165,572,2003.666667,2020.0,2020.0,0.825,0.755,0.89,1,1,3,2,1,0,0,0,0,0.000000,70.0,0,0,1,30.0,65.0,35.0,8,0,0,0,1,1,0
4,H3546,NOVOTEL PARIS CENTRE TOUR EIFFEL,2.282836,48.849778,764,1976.000000,2020.0,2020.0,0.825,0.755,0.89,1,3,36,1,1,0,0,1,1,0.000000,70.0,0,0,1,30.0,65.0,35.0,7,0,0,0,1,0,1
5,H0373,MERCURE MONTMARTRE SACRE COEUR,2.329923,48.885048,305,2003.666667,2020.0,2020.0,0.825,0.755,0.89,0,1,1,1,0,1,0,0,0,0.000000,70.0,1,0,0,30.0,65.0,35.0,6,0,0,1,0,0,1
6,H6188,MERCURE PARIS BOULOGNE,2.256274,48.833827,191,2003.666667,2020.0,2020.0,0.825,0.755,0.89,1,2,13,1,1,1,0,1,0,0.000000,70.0,1,0,0,30.0,65.0,35.0,6,0,0,1,0,0,1


In [37]:
df_poi = pd.read_csv("poi.csv")

# Types Food & Beverage
fb_types = [
    "convenience", "bakery", "supermarket", "alcohol",
    "confectionery", "beverages", "grocery", "ice_cream"
]

# Types Non Food & Beverage
non_fb_types = [
    "cosmetics", "gift", "tobacco", "kiosk"
]

df_poi["is_fb"] = df_poi["SHOP_TYPE"].isin(fb_types)
df_poi["is_not_fb"] = df_poi["SHOP_TYPE"].isin(non_fb_types)

df_poi_clean = (
    df_poi.groupby(["HOTEL_NAME", "HOTEL_CITY", "HOTEL_LAT", "HOTEL_LON"])
    .apply(lambda x: pd.Series({
        "fb_0_1km": ((x["DIST_KM"] <= 0.1) & x["is_fb"]).sum(),
        "fb_0_2km": ((x["DIST_KM"] <= 0.2) & x["is_fb"]).sum(),
        "fb_0_3km": ((x["DIST_KM"] <= 0.3) & x["is_fb"]).sum(),
        "fb_0_4km": ((x["DIST_KM"] <= 0.4) & x["is_fb"]).sum(),
        "fb_0_5km": ((x["DIST_KM"] <= 0.5) & x["is_fb"]).sum(),
        "not_fb_0_1km": ((x["DIST_KM"] <= 0.1) & x["is_not_fb"]).sum(),
        "not_fb_0_2km": ((x["DIST_KM"] <= 0.2) & x["is_not_fb"]).sum(),
        "not_fb_0_3km": ((x["DIST_KM"] <= 0.3) & x["is_not_fb"]).sum(),
        "not_fb_0_4km": ((x["DIST_KM"] <= 0.4) & x["is_not_fb"]).sum(),
        "not_fb_0_5km": ((x["DIST_KM"] <= 0.5) & x["is_not_fb"]).sum(),
    }))
    .reset_index()
)

/tmp/ipykernel_21865/65113361.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


In [38]:
df_poi_clean

,HOTEL_NAME,HOTEL_CITY,HOTEL_LAT,HOTEL_LON,fb_0_1km,fb_0_2km,fb_0_3km,fb_0_4km,fb_0_5km,not_fb_0_1km,not_fb_0_2km,not_fb_0_3km,not_fb_0_4km,not_fb_0_5km
0,Ibis budget Nice,Nice,43.667571,7.214308,0,0,0,3,3,3,3,6,6,6
1,Ibis budget Strasbourg Centre République,Strasbourg,48.585058,7.736591,0,30,60,102,120,0,3,3,24,36
2,Mercure Paris Montmartre Sacré-Cœur,Paris,48.885240,2.330055,6,12,57,123,195,3,3,27,45,54
3,Novotel Megève Mont-Blanc,Megève,45.859850,6.619478,3,3,6,6,15,0,0,3,3,3
4,Novotel Paris Tour Eiffel,Paris,48.849660,2.283704,3,6,9,36,69,0,9,12,15,18
5,Novotel Porte d’Italie,Paris,48.819054,2.359406,9,18,21,42,75,3,3,3,6,9


In [17]:
df_weather = pd.read_csv("weather.csv")

df_weather["time"] = pd.to_datetime(df_weather["time"])
df_weather["mois"] = df_weather["time"].dt.to_period("M").astype(str)

cols_stats = ["temp", "dwpt", "rhum", "prcp", "snow", "wspd", "wpgt", "pres", "tsun"]

df_weather_clean = (
    df_weather
    .groupby(["lat", "lon", "mois"])[cols_stats]
    .agg([
        "min",
        "max",
        "mean",
        "median",
        lambda x: x.quantile(0.25),
        lambda x: x.quantile(0.75),
    ])
)

df_weather_clean.columns = [
    f"{col}_{stat}"
    for col, stat in df_weather_clean.columns
]

df_weather_clean = df_weather_clean.reset_index()

# Renommer les lambdas
df_weather_clean.columns = (
    df_weather_clean.columns
    .str.replace("<lambda_0>", "q25")
    .str.replace("<lambda_1>", "q75")
)


# Reset index
df_weather_clean = df_weather_clean.reset_index()

# Renommer les lambdas
df_weather_clean.columns = (
    df_weather_clean.columns
    .str.replace("<lambda_0>", "q25")
    .str.replace("<lambda_1>", "q75")
)

# Pivot : une seule ligne par lat/lon
df_weather_clean = df_weather_clean.pivot(
    index=["lat", "lon"],
    columns="mois"
)

# Aplatir les colonnes
df_weather_clean.columns = [
    f"m{pd.to_datetime(month).month:02d}_{col}"
    for col, month in df_weather_clean.columns
]

# Revenir en dataframe classique
df_weather_clean = df_weather_clean.reset_index()

# Trier les colonnes proprement par mois puis nom de feature

ordered_cols = ["lat", "lon"] + sorted(
    [c for c in df_weather_clean.columns if c not in ["lat", "lon"]],
    key=lambda x: (
        int(x.split("_")[0][1:]),  # numéro du mois
        x.split("_", 1)[1]         # reste du nom
    )
)

df_weather_clean = df_weather_clean[ordered_cols]
df_weather_clean.rename(columns={"lat": "HOTEL_LAT", "lon" : "HOTEL_LON"}, inplace=True)


In [18]:
df_weather_clean

HOTEL_LAT  HOTEL_LON  m01_dwpt_max  m01_dwpt_max  m01_dwpt_max  \
0  43.667571   7.214308          11.7          13.8          10.1   
1  45.859850   6.619478           7.8           7.3           2.1   
2  48.585058   7.736591          10.9          10.1           8.1   
3  48.849660   2.283704          12.0          10.2           8.9   
4  48.885240   2.330055          12.0          10.2           8.9   

   m01_dwpt_max  m01_dwpt_max  m01_dwpt_max  m01_dwpt_max  m01_dwpt_max  \
0          11.7          13.8          10.1          11.7          13.8   
1           7.8           7.3           2.1           7.8           7.3   
2          10.9          10.1           8.1          10.9          10.1   
3          12.0          10.2           8.9          12.0          10.2   
4          12.0          10.2           8.9          12.0          10.2   

   m01_dwpt_max  m01_dwpt_mean  m01_dwpt_mean  m01_dwpt_mean  m01_dwpt_mean  \
0          10.1       4.190188       4.944220       2.316801       4.190188   
1           2.1      -4.595161      -5.025941      -6.830242      -4.595161   
2           8.1       0.284812       0.931855      -0.064382       0.284812   
3           8.9       1.649194       2.165860       2.820565       1.649194   
4           8.9       1.649194       2.165860       2.820565       1.649194   

   m01_dwpt_mean  m01_dwpt_mean  m01_dwpt_mean  m01_dwpt_mean  m01_dwpt_mean  \
0       4.944220       2.316801       4.190188       4.944220       2.316801   
1      -5.025941      -6.830242      -4.595161      -5.025941      -6.830242   
2       0.931855      -0.064382       0.284812       0.931855      -0.064382   
3       2.165860       2.820565       1.649194       2.165860       2.820565   
4       2.165860       2.820565       1.649194       2.165860       2.820565   

   m01_dwpt_median  m01_dwpt_median  m01_dwpt_median  m01_dwpt_median  \
0             4.85              6.0              3.1             4.85   
1            -3.00             -3.4             -4.8            -3.00   
2             0.00              0.1              0.0             0.00   
3             0.95              1.9              3.5             0.95   
4             0.95              1.9              3.5             0.95   

   m01_dwpt_median  m01_dwpt_median  m01_dwpt_median  m01_dwpt_median  \
0              6.0              3.1             4.85              6.0   
1             -3.4             -4.8            -3.00             -3.4   
2              0.1              0.0             0.00              0.1   
3              1.9              3.5             0.95              1.9   
4              1.9              3.5             0.95              1.9   

   m01_dwpt_median  m01_dwpt_min  m01_dwpt_min  m01_dwpt_min  m01_dwpt_min  \
0              3.1          -7.7          -7.8         -11.7          -7.7   
1             -4.8         -29.7         -23.9         -27.9         -29.7   
2              0.0          -9.8          -6.0         -11.1          -9.8   
3              3.5          -7.9          -5.2          -6.5          -7.9   
4              3.5          -7.9          -5.2          -6.5          -7.9   

   m01_dwpt_min  m01_dwpt_min  m01_dwpt_min  m01_dwpt_min  m01_dwpt_min  \
0          -7.8         -11.7          -7.7          -7.8         -11.7   
1         -23.9         -27.9         -29.7         -23.9         -27.9   
2          -6.0         -11.1          -9.8          -6.0         -11.1   
3          -5.2          -6.5          -7.9          -5.2          -6.5   
4          -5.2          -6.5          -7.9          -5.2          -6.5   

   m01_dwpt_q25  m01_dwpt_q25  m01_dwpt_q25  m01_dwpt_q25  m01_dwpt_q25  \
0          2.10           2.7        -0.900          2.10           2.7   
1         -7.65          -9.5       -10.325         -7.65          -9.5   
2         -4.30          -1.4        -2.000         -4.30          -1.4   
3         -3.20          -0.6         0.200         -3.20          -0.6   
4       

In [8]:
import pandas as pd

df = pd.read_csv("data.csv")

df["DATETIME"] = pd.to_datetime(df["DATETIME"])
df["mois"] = df["DATETIME"].dt.month

# Montant de vente
df["montant_vente"] = df["PRIX TTC"] * df["QUANTITE"]

# Infos hôtel à garder
hotel_cols = [
    "HOTEL_NAME",
    "HOTEL_CITY",
    "HOTEL_LAT",
    "HOTEL_LON",
]

# Agrégation par hôtel / mois / type
monthly = (
    df
    .groupby(hotel_cols + ["mois", "TYPE"])
    .agg(
        montant=("montant_vente", "sum"),
        nbr_ventes=("ORDER ID (TICKET DE CAISSE)", "nunique")
    )
    .reset_index()
)

# Passage en colonnes
wide = monthly.pivot_table(
    index=hotel_cols,
    columns=["mois", "TYPE"],
    values=["montant", "nbr_ventes"],
    fill_value=0
)

# Aplatir les noms de colonnes
wide.columns = [
    f"m{mois:02d}_{type_}_{metric}"
    .replace("&", "")
    .replace("-", "_")
    .replace(" ", "_")
    for metric, mois, type_ in wide.columns
]

df_hotel_monthly = wide.reset_index()

# Trier les colonnes : infos hôtel puis m01, m02, ...
hotel_cols_final = hotel_cols

monthly_cols = sorted(
    [c for c in df_hotel_monthly.columns if c not in hotel_cols_final],
    key=lambda x: (
        int(x.split("_")[0][1:]),  # mois
        x
    )
)

df_hotel_monthly = df_hotel_monthly[hotel_cols_final + monthly_cols]

In [9]:
df_hotel_monthly

,HOTEL_NAME,HOTEL_CITY,HOTEL_LAT,HOTEL_LON,m01_FB_montant,m01_FB_nbr_ventes,m01_NON_FB_montant,m01_NON_FB_nbr_ventes,m02_FB_montant,m02_FB_nbr_ventes,m02_NON_FB_montant,m02_NON_FB_nbr_ventes,m03_FB_montant,m03_FB_nbr_ventes,m03_NON_FB_montant,m03_NON_FB_nbr_ventes,m04_FB_montant,m04_FB_nbr_ventes,m04_NON_FB_montant,m04_NON_FB_nbr_ventes,m05_FB_montant,m05_FB_nbr_ventes,m05_NON_FB_montant,m05_NON_FB_nbr_ventes,m06_FB_montant,m06_FB_nbr_ventes,m06_NON_FB_montant,m06_NON_FB_nbr_ventes,m07_FB_montant,m07_FB_nbr_ventes,m07_NON_FB_montant,m07_NON_FB_nbr_ventes,m08_FB_montant,m08_FB_nbr_ventes,m08_NON_FB_montant,m08_NON_FB_nbr_ventes,m09_FB_montant,m09_FB_nbr_ventes,m09_NON_FB_montant,m09_NON_FB_nbr_ventes,m10_FB_montant,m10_FB_nbr_ventes,m10_NON_FB_montant,m10_NON_FB_nbr_ventes,m11_FB_montant,m11_FB_nbr_ventes,m11_NON_FB_montant,m11_NON_FB_nbr_ventes,m12_FB_montant,m12_FB_nbr_ventes,m12_NON_FB_montant,m12_NON_FB_nbr_ventes
0,Ibis budget Nice,Nice,43.667571,7.214308,557.5,146.0,132.0,9.0,470.5,131.0,26.00,3.0,1014.6,301.0,215.0,19.0,640.6,187.0,405.10,28.0,975.3,262.0,639.0,50.0,1460.1,396.0,1003.10,79.0,1661.20,438.0,1081.5,81.0,1762.8,451.0,1678.00,118.0,1368.8,367.0,1231.0,107.0,1102.3,292.0,696.00,59.0,672.3,190.0,123.00,13.0,569.9,152.0,89.0,8.0
1,Ibis budget Strasbourg Centre République,Strasbourg,48.585058,7.736591,1720.5,325.0,0.0,0.0,822.0,129.0,0.00,0.0,1461.5,306.0,0.0,0.0,1447.0,241.0,0.00,0.0,654.0,142.0,0.0,0.0,1709.0,311.0,0.00,0.0,1400.50,239.0,0.0,0.0,1723.5,347.0,0.00,0.0,636.0,162.0,0.0,0.0,1541.5,310.0,0.00,0.0,2117.5,382.0,0.00,0.0,3364.0,641.0,0.0,0.0
2,Mercure Paris Montmartre Sacré-Cœur,Paris,48.885240,2.330055,8704.0,623.0,188.3,17.0,8998.5,820.0,719.00,21.0,11098.0,1053.0,474.0,25.0,10201.0,924.0,739.00,27.0,7562.0,619.0,355.0,19.0,8693.0,833.0,406.00,13.0,6425.00,572.0,155.0,9.0,6680.0,467.0,172.00,8.0,11380.0,849.0,156.0,6.0,13398.0,962.0,186.50,10.0,8681.0,652.0,146.50,11.0,8493.0,653.0,313.3,21.0
3,Novotel Megève Mont-Blanc,Megève,45.859850,6.619478,1705.4,233.0,2848.7,136.0,1338.0,233.0,2918.55,162.0,954.5,171.0,2291.5,132.0,461.5,65.0,1305.55,66.0,473.4,62.0,1503.2,55.0,665.5,80.0,1491.85,72.0,1013.50,151.0,2123.1,101.0,1232.5,169.0,3411.45,166.0,522.3,80.0,2253.9,95.0,441.8,49.0,806.55,37.0,437.9,61.0,1913.75,83.0,1407.8,224.0,2480.3,121.0
4,Novotel Paris Tour Eiffel,Paris,48.849660,2.283704,18282.2,2256.0,3858.0,387.0,18028.7,2268.0,5220.00,528.0,18779.4,2469.0,4822.0,548.0,15211.1,1849.0,2825.00,303.0,12578.5,1549.0,2696.0,283.0,16949.0,1869.0,3986.00,358.0,21624.67,2230.0,4167.5,353.0,14124.2,1644.0,3553.00,336.0,11643.4,1526.0,3841.0,373.0,20319.3,2728.0,7184.00,698.0,17429.4,2362.0,6300.00,640.0,22327.4,2857.0,5782.0,596.0


In [65]:
# Arrondi des coordonnées
df_poi_clean["HOTEL_LAT"] = df_poi_clean["HOTEL_LAT"].round(5)
df_poi_clean["HOTEL_LON"] = df_poi_clean["HOTEL_LON"].round(5)

df_weather_clean["HOTEL_LAT"] = df_weather_clean["HOTEL_LAT"].round(5)
df_weather_clean["HOTEL_LON"] = df_weather_clean["HOTEL_LON"].round(5)

df_rod_clean["HOTEL_LAT"] = df_rod_clean["HOTEL_LAT"].round(5)
df_rod_clean["HOTEL_LON"] = df_rod_clean["HOTEL_LON"].round(5)

df_hotel_monthly["HOTEL_LAT"] = df_hotel_monthly["HOTEL_LAT"].round(5)
df_hotel_monthly["HOTEL_LON"] = df_hotel_monthly["HOTEL_LON"].round(5)

df_rod_clean["HOTEL_NAME"] = df_rod_clean["HOTEL_NAME"].replace({
    "IBIS BUDGET NICE CALIFORNIE" : "Ibis budget Nice",
    "IBIS BUDGET STRASBOURG REPUBLIQUE" : "Ibis budget Strasbourg Centre République",
    "NOVOTEL MEGEVE MONT BLANC" : "Novotel Megève Mont-Blanc",
    "NOVOTEL PARIS CENTRE TOUR EIFFEL": "Novotel Paris Tour Eiffel",
    "MERCURE MONTMARTRE SACRE COEUR": "Mercure Paris Montmartre Sacré-Cœur",
})

df_rod_clean = df_rod_clean.drop(["HOTEL_LAT", "HOTEL_LON"], axis = 1)

# =========================
# Merge POI + WEATHER
# =========================

df_final = df_poi_clean.merge(
    df_weather_clean,
    on=["HOTEL_LAT", "HOTEL_LON"],
    how="inner"
)



# =========================
# Merge avec SALES
# =========================

df_final = df_final.merge(
    df_hotel_monthly,
    on=["HOTEL_NAME", "HOTEL_CITY", "HOTEL_LAT", "HOTEL_LON"],
    how="inner"
)

# =========================
# Merge avec ROD
# =========================
df_final = df_final.merge(
    df_rod_clean,
    on=["HOTEL_NAME"],
    how="inner"
)


In [66]:
df_final

HOTEL_NAME  HOTEL_CITY  HOTEL_LAT  HOTEL_LON  \
0                          Ibis budget Nice        Nice   43.66757    7.21431   
1  Ibis budget Strasbourg Centre République  Strasbourg   48.58506    7.73659   
2       Mercure Paris Montmartre Sacré-Cœur       Paris   48.88524    2.33006   
3                 Novotel Megève Mont-Blanc      Megève   45.85985    6.61948   
4                 Novotel Paris Tour Eiffel       Paris   48.84966    2.28370   

   fb_0_1km  fb_0_2km  fb_0_3km  fb_0_4km  fb_0_5km  not_fb_0_1km  \
0         0         0         0         3         3             3   
1         0        30        60       102       120             0   
2         6        12        57       123       195             3   
3         3         3         6         6        15             0   
4         3         6         9        36        69             0   

   not_fb_0_2km  not_fb_0_3km  not_fb_0_4km  not_fb_0_5km  m01_dwpt_max  \
0             3             6             6             6          11.7   
1             3             3            24            36          10.9   
2             3            27            45            54          12.0   
3             0             3             3             3           7.8   
4             9            12            15            18          12.0   

   m01_dwpt_max  m01_dwpt_max  m01_dwpt_max  m01_dwpt_max  m01_dwpt_max  \
0          13.8          10.1          11.7          13.8          10.1   
1          10.1           8.1          10.9          10.1           8.1   
2          10.2           8.9          12.0          10.2           8.9   
3           7.3           2.1           7.8           7.3           2.1   
4          10.2           8.9          12.0          10.2           8.9   

   m01_dwpt_max  m01_dwpt_max  m01_dwpt_max  m01_dwpt_mean  m01_dwpt_mean  \
0          11.7          13.8          10.1       4.190188       4.944220   
1          10.9          10.1           8.1       0.284812       0.931855   
2          12.0          10.2           8.9       1.649194       2.165860   
3           7.8           7.3           2.1      -4.595161      -5.025941   
4          12.0          10.2           8.9       1.649194       2.165860   

   m01_dwpt_mean  m01_dwpt_mean  m01_dwpt_mean  m01_dwpt_mean  m01_dwpt_mean  \
0       2.316801       4.190188       4.944220       2.316801       4.190188   
1      -0.064382       0.284812       0.931855      -0.064382       0.284812   
2       2.820565       1.649194       2.165860       2.820565       1.649194   
3      -6.830242      -4.595161      -5.025941      -6.830242      -4.595161   
4       2.820565       1.649194       2.165860       2.820565       1.649194   

   m01_dwpt_mean  m01_dwpt_mean  m01_dwpt_median  m01_dwpt_median  \
0       4.944220       2.316801             4.85              6.0   
1       0.931855      -0.064382             0.00              0.1   
2       2.165860       2.820565             0.95              1.9   
3      -5.025941      -6.830242            -3.00             -3.4   
4       2.165860       2.820565             0.95              1.9   

   m01_dwpt_median  m01_dwpt_median  m01_dwpt_median  m01_dwpt_median  \
0              3.1             4.85              6.0              3.1   
1              0.0             0.00              0.1              0.0   
2              3.5             0.95              1.9              3.5   
3             -4.8            -3.00             -3.4             -4.8   
4              3.5             0.95              1.9              3.5   

   m01_dwpt_median  m01_dwpt_median  m01_dwpt_median  m01_dwpt_min  \
0             4.85              6.0              3.1          -7.7   
1             0.00              0.1              0.0          -9.8   
2             0.95              1.9              3.5          -7.9   
3            -3.00             -3.4             -4.8         -29.7   
4             0.95              1.9              3.5          -7.9   

   m01_dwp

In [91]:
df_final

HOTEL_NAME  HOTEL_CITY  HOTEL_LAT  HOTEL_LON  \
0                          Ibis budget Nice        Nice   43.66757    7.21431   
1  Ibis budget Strasbourg Centre République  Strasbourg   48.58506    7.73659   
2       Mercure Paris Montmartre Sacré-Cœur       Paris   48.88524    2.33006   
3                 Novotel Megève Mont-Blanc      Megève   45.85985    6.61948   
4                 Novotel Paris Tour Eiffel       Paris   48.84966    2.28370   

   fb_0_1km  fb_0_2km  fb_0_3km  fb_0_4km  fb_0_5km  not_fb_0_1km  \
0         0         0         0         3         3             3   
1         0        30        60       102       120             0   
2         6        12        57       123       195             3   
3         3         3         6         6        15             0   
4         3         6         9        36        69             0   

   not_fb_0_2km  not_fb_0_3km  not_fb_0_4km  not_fb_0_5km  m01_dwpt_max  \
0             3             6             6             6          11.7   
1             3             3            24            36          10.9   
2             3            27            45            54          12.0   
3             0             3             3             3           7.8   
4             9            12            15            18          12.0   

   m01_dwpt_max  m01_dwpt_max  m01_dwpt_max  m01_dwpt_max  m01_dwpt_max  \
0          13.8          10.1          11.7          13.8          10.1   
1          10.1           8.1          10.9          10.1           8.1   
2          10.2           8.9          12.0          10.2           8.9   
3           7.3           2.1           7.8           7.3           2.1   
4          10.2           8.9          12.0          10.2           8.9   

   m01_dwpt_max  m01_dwpt_max  m01_dwpt_max  m01_dwpt_mean  m01_dwpt_mean  \
0          11.7          13.8          10.1       4.190188       4.944220   
1          10.9          10.1           8.1       0.284812       0.931855   
2          12.0          10.2           8.9       1.649194       2.165860   
3           7.8           7.3           2.1      -4.595161      -5.025941   
4          12.0          10.2           8.9       1.649194       2.165860   

   m01_dwpt_mean  m01_dwpt_mean  m01_dwpt_mean  m01_dwpt_mean  m01_dwpt_mean  \
0       2.316801       4.190188       4.944220       2.316801       4.190188   
1      -0.064382       0.284812       0.931855      -0.064382       0.284812   
2       2.820565       1.649194       2.165860       2.820565       1.649194   
3      -6.830242      -4.595161      -5.025941      -6.830242      -4.595161   
4       2.820565       1.649194       2.165860       2.820565       1.649194   

   m01_dwpt_mean  m01_dwpt_mean  m01_dwpt_median  m01_dwpt_median  \
0       4.944220       2.316801             4.85              6.0   
1       0.931855      -0.064382             0.00              0.1   
2       2.165860       2.820565             0.95              1.9   
3      -5.025941      -6.830242            -3.00             -3.4   
4       2.165860       2.820565             0.95              1.9   

   m01_dwpt_median  m01_dwpt_median  m01_dwpt_median  m01_dwpt_median  \
0              3.1             4.85              6.0              3.1   
1              0.0             0.00              0.1              0.0   
2              3.5             0.95              1.9              3.5   
3             -4.8            -3.00             -3.4             -4.8   
4              3.5             0.95              1.9              3.5   

   m01_dwpt_median  m01_dwpt_median  m01_dwpt_median  m01_dwpt_min  \
0             4.85              6.0              3.1          -7.7   
1             0.00              0.1              0.0          -9.8   
2             0.95              1.9              3.5          -7.9   
3            -3.00             -3.4             -4.8         -29.7   
4             0.95              1.9              3.5          -7.9   

   m01_dwp

In [111]:
ml_skip_columns= ["HOTEL_CODE", "HOTEL_NAME", "HOTEL_CITY", "HOTEL_LAT", "HOTEL_LON"]

pattern = r"^m\d{2}_.*(vente|montant).*"
ml_target_variables = list(set([
    col for col in df_final.columns
    if(bool(re.match(pattern, col)))
]))

ml_desc_variables = list(set([col for col in df_final.columns  if ((col not in ml_skip_columns) and (col not in ml_target_variables))]))

X = df_final[ml_desc_variables].fillna(0)
y = df_final[ml_target_variables]

In [112]:
m = keras.models.Sequential([
    keras.layers.InputLayer(shape = [X.shape[1]]),
    keras.layers.Dense(y.shape[1])
])

m.compile(loss = "mse", optimizer = "adam")

In [114]:
m.fit(X, y, batch_size = 1, epochs = 1000)

Epoch 1/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 9434291.0000  
Epoch 2/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9761992.0000 
Epoch 3/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 9345672.0000 
Epoch 4/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9372672.0000 
Epoch 5/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9486747.0000  
Epoch 6/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9247579.0000 
Epoch 7/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 9324866.0000 
Epoch 8/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 9192097.0000 
Epoch 9/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 9186263.0000 
Epoch 10/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 9144471.0000 
Epoch 11/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 9247017.0000 
Epoch 12/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9089150.0000  
Epoch 13/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 9309483.0000 
Epoch 14/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss

In [116]:
m.fit(X, y, batch_size = 1, epochs = 10000)

Epoch 1/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1262.9412
Epoch 2/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2453.7954
Epoch 3/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1899.1707 
Epoch 4/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 2457.7524 
Epoch 5/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 2168.5422
Epoch 6/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 745.8651
Epoch 7/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 714.5196
Epoch 8/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 992.8602 
Epoch 9/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 843.5859 
Epoch 10/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1103.8976
Epoch 11/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 858.4625
Epoch 12/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 774.7701 
Epoch 13/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 857.3138
Epoch 14/10000
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 1452.4352
Epoch 15/10000
5/5 ━━━